# Entrenamiento de ejemplo con FLAML para forecasting
Este notebook genera una serie temporal sintética, entrena un modelo AutoML y exporta artefactos compatibles con el pipeline de inferencia (modelo .pickle, requirements y esquema de datos).

In [ ]:
!pip install -q flaml pandas scikit-learn

In [ ]:
import os
import pickle
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from flaml import AutoML

MODEL_NAME = "sample_forecast"  # Cambiar según el modelo
ARTIFACT_ROOT = Path("artifacts") / MODEL_NAME
(ARTIFACT_ROOT / "model").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "requirements").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "data").mkdir(parents=True, exist_ok=True)

np.random.seed(42)
date_range = pd.date_range(datetime.now() - timedelta(days=200), periods=200, freq="D")
signal = np.sin(np.linspace(0, 20, 200)) + np.random.normal(0, 0.1, 200)
df = pd.DataFrame({"ds": date_range, "y": signal})
df["dayofweek"] = df["ds"].dt.dayofweek

train_df = df.iloc[:-14]
test_df = df.iloc[-14:]

automl = AutoML()
settings = {
    "time_budget": 60,
    "task": "forecast",
    "log_file_name": "automl.log",
    "metric": "mse",
    "eval_method": "holdout",
    "split_ratio": 0.2,
    "period": 7,
    "label": "y",
    "time_col": "ds",
    "estimator_list": ["lgbm", "xgboost"],
}

automl.fit(dataframe=train_df, **settings)
print("Mejor modelo:", automl.best_estimator)
print("Mejor configuración:", automl.best_config)

# Evaluación simple
preds = automl.predict(test_df)
print("MSE de validación:", np.mean((preds - test_df['y'].values) ** 2))

# Guardar modelo
model_path = ARTIFACT_ROOT / "model" / f"{MODEL_NAME}.pickle"
with open(model_path, "wb") as f:
    pickle.dump(automl, f)
print("Modelo guardado en", model_path)

# Generar requirements mínimos
requirements_path = ARTIFACT_ROOT / "requirements" / "requirements.txt"
requirements_path.write_text("flaml
scikit-learn
pandas")
print("Requirements escritos en", requirements_path)

# Crear esquema de datos de entrada vacío
schema_path = ARTIFACT_ROOT / "data" / "schema.parquet"
schema_stub = pd.DataFrame(columns=train_df.columns)
schema_stub.to_parquet(schema_path, index=False)
print("Esquema vacío guardado en", schema_path)
